# BioReview SFT: Model Comparison Analysis

Publication-quality analysis of QLoRA SFT fine-tuned models for automated peer review concern generation.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from collections import Counter

# ── Publication style ──────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ── Color palette (Nature-style, colorblind-safe) ─────────────
COLORS = {
    'Qwen2.5-7B':    '#4E79A7',  # steel blue
    'Qwen3-8B':      '#59A14F',  # green
    'Qwen3.5-9B':    '#E15759',  # coral red
    'Qwen2.5-14B':   '#F28E2B',  # orange
    'DeepSeek-R1-14B':'#76B7B2', # teal
    'GPT-4o-mini':   '#B07AA1',  # purple (baseline)
    'Gemini-2.5-Flash':'#9C755F',# brown (baseline)
}

RESULTS_DIR = Path('../results/sft_eval')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## 1. Load Results

In [ ]:
# ── Load summary JSONs ────────────────────────────────────────
MODEL_DISPLAY = {
    'qwen7b_bioreview_v1':          'Qwen2.5-7B',
    'qwen3_8b_bioreview_v1':        'Qwen3-8B',
    'qwen3.5_9b_bioreview_v1':      'Qwen3.5-9B',
    'qwen2.5_14b_bioreview_v1':     'Qwen2.5-14B',
    'deepseek_r1_14b_bioreview_v1': 'DeepSeek-R1-14B',
}

rows = []
for p in sorted(RESULTS_DIR.glob('*.summary.json')):
    data = json.loads(p.read_text())
    m = data.get('eval_metrics', {})
    if not m:
        continue
    model_key = p.stem.replace('_val.summary', '')
    rows.append({
        'model': MODEL_DISPLAY.get(model_key, model_key),
        'model_key': model_key,
        'precision': m.get('precision_overall', 0),
        'recall': m.get('recall_overall', 0),
        'f1': m.get('f1_micro', 0),
        'recall_major': m.get('recall_major', 0),
        'n_articles': m.get('n_articles', 0),
        'n_tool_concerns': m.get('n_tool_concerns', 0),
        'n_human_concerns': m.get('n_human_concerns', 0),
        'n_zero_recall': m.get('n_zero_recall_articles', 0),
        'processed': data.get('processed', 0),
        'failed_parse': data.get('failed_parse', 0),
        'total_time': data.get('total_time', 0),
    })

# ── Add baselines ─────────────────────────────────────────────
rows.append({
    'model': 'GPT-4o-mini', 'model_key': 'gpt4o_mini_baseline',
    'precision': 0.7531, 'recall': 0.6472, 'f1': 0.6962,
    'recall_major': None, 'n_articles': 982,
    'n_tool_concerns': None, 'n_human_concerns': 13957,
    'n_zero_recall': None, 'processed': 982, 'failed_parse': 0,
    'total_time': None,
})
rows.append({
    'model': 'Gemini-2.5-Flash', 'model_key': 'gemini_baseline',
    'precision': 0.8820, 'recall': 0.3011, 'f1': 0.4489,
    'recall_major': None, 'n_articles': 982,
    'n_tool_concerns': None, 'n_human_concerns': 13957,
    'n_zero_recall': None, 'processed': 982, 'failed_parse': 0,
    'total_time': None,
})

df = pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)
df

## 2. Load Per-Article JSONL for Detailed Analysis

In [ ]:
# ── Load per-article predictions ──────────────────────────────
article_data = {}
for p in sorted(RESULTS_DIR.glob('*_val.jsonl')):
    model_key = p.stem.replace('_val', '')
    display = MODEL_DISPLAY.get(model_key, model_key)
    records = []
    with open(p) as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    article_data[display] = records
    print(f'{display}: {len(records)} articles, '
          f'{sum(len(r.get("structured_concerns", [])) for r in records)} concerns')

## 3. Figure 1: F1 / Precision / Recall Comparison

In [ ]:
def fig_metric_comparison(df, save_path=None):
    """Grouped bar chart: F1, Precision, Recall for each model."""
    metrics = ['f1', 'precision', 'recall']
    labels = ['F1 (micro)', 'Precision', 'Recall']
    
    # Sort by F1
    plot_df = df.sort_values('f1', ascending=True).copy()
    models = plot_df['model'].tolist()
    
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    
    y = np.arange(len(models))
    bar_h = 0.25
    
    for i, (metric, label) in enumerate(zip(metrics, labels)):
        vals = plot_df[metric].fillna(0).values
        bars = ax.barh(y + i * bar_h, vals, height=bar_h, label=label,
                       color=['#4E79A7', '#E15759', '#59A14F'][i],
                       edgecolor='white', linewidth=0.3)
    
    ax.set_yticks(y + bar_h)
    ax.set_yticklabels(models)
    ax.set_xlabel('Score')
    ax.set_xlim(0, 1.0)
    ax.legend(loc='lower right', frameon=False)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path)
        fig.savefig(save_path.with_suffix('.pdf'))
    plt.show()

fig_metric_comparison(df, FIGURES_DIR / 'fig1_metric_comparison.png')

## 4. Figure 2: Precision vs. Recall Scatter

In [ ]:
def fig_precision_recall(df, save_path=None):
    """Precision vs Recall scatter with F1 iso-lines."""
    fig, ax = plt.subplots(figsize=(3.5, 3.0))
    
    # F1 iso-lines
    for f1_val in [0.2, 0.4, 0.6, 0.8]:
        r_range = np.linspace(0.01, 1.0, 200)
        p_range = (f1_val * r_range) / (2 * r_range - f1_val)
        mask = (p_range > 0) & (p_range <= 1)
        ax.plot(r_range[mask], p_range[mask], '--', color='#CCCCCC',
                linewidth=0.5, zorder=1)
        # Label at the curve edge
        idx = np.argmin(np.abs(p_range - 0.98))
        if mask[idx]:
            ax.text(r_range[idx], 0.99, f'F1={f1_val}',
                    fontsize=6, color='#999999', ha='center', va='bottom')
    
    for _, row in df.iterrows():
        color = COLORS.get(row['model'], '#333333')
        is_baseline = 'baseline' in row.get('model_key', '')
        marker = 's' if is_baseline else 'o'
        size = 30 if is_baseline else 50
        
        ax.scatter(row['recall'], row['precision'], c=color,
                   s=size, marker=marker, edgecolors='white',
                   linewidths=0.5, zorder=3)
        
        # Label
        offset_x = 0.02 if row['recall'] < 0.5 else -0.02
        ha = 'left' if row['recall'] < 0.5 else 'right'
        ax.annotate(row['model'], (row['recall'], row['precision']),
                    xytext=(offset_x, 0.02), textcoords='offset fontsize',
                    fontsize=6, ha=ha, va='bottom', color=color)
    
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.08)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path)
        fig.savefig(save_path.with_suffix('.pdf'))
    plt.show()

fig_precision_recall(df, FIGURES_DIR / 'fig2_precision_recall.png')

## 5. Figure 3: Concerns per Article Distribution

In [ ]:
def fig_concerns_distribution(article_data, save_path=None):
    """Box/violin plot of number of concerns per article."""
    plot_data = []
    model_order = []
    
    for model_name, records in article_data.items():
        counts = [len(r.get('structured_concerns', [])) for r in records]
        if not counts or max(counts) == 0:
            continue
        plot_data.append(counts)
        model_order.append(model_name)
    
    if not plot_data:
        print('No data with non-zero concerns.')
        return
    
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    
    parts = ax.violinplot(plot_data, positions=range(len(model_order)),
                          showmeans=True, showmedians=True, showextrema=False)
    
    for i, pc in enumerate(parts['bodies']):
        color = COLORS.get(model_order[i], '#4E79A7')
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    parts['cmeans'].set_color('#333333')
    parts['cmedians'].set_color('#E15759')
    
    ax.set_xticks(range(len(model_order)))
    ax.set_xticklabels(model_order, rotation=30, ha='right')
    ax.set_ylabel('Concerns per article')
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path)
        fig.savefig(save_path.with_suffix('.pdf'))
    plt.show()

fig_concerns_distribution(article_data, FIGURES_DIR / 'fig3_concerns_distribution.png')

## 6. Figure 4: Severity Distribution

In [ ]:
def fig_severity_distribution(article_data, save_path=None):
    """Stacked bar: major / minor / optional per model."""
    severity_order = ['major', 'minor', 'optional']
    sev_colors = {'major': '#E15759', 'minor': '#F28E2B', 'optional': '#76B7B2'}
    
    model_names = []
    sev_counts = {s: [] for s in severity_order}
    
    for model_name, records in article_data.items():
        all_concerns = [c for r in records for c in r.get('structured_concerns', [])]
        if not all_concerns:
            continue
        total = len(all_concerns)
        counter = Counter(c.get('severity', 'unknown') for c in all_concerns)
        model_names.append(model_name)
        for s in severity_order:
            sev_counts[s].append(counter.get(s, 0) / total if total > 0 else 0)
    
    if not model_names:
        print('No severity data.')
        return
    
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    x = np.arange(len(model_names))
    bottom = np.zeros(len(model_names))
    
    for sev in severity_order:
        vals = np.array(sev_counts[sev])
        ax.bar(x, vals, bottom=bottom, label=sev.capitalize(),
               color=sev_colors[sev], edgecolor='white', linewidth=0.3,
               width=0.6)
        bottom += vals
    
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=30, ha='right')
    ax.set_ylabel('Proportion')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_ylim(0, 1.05)
    ax.legend(loc='upper right', frameon=False)
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path)
        fig.savefig(save_path.with_suffix('.pdf'))
    plt.show()

fig_severity_distribution(article_data, FIGURES_DIR / 'fig4_severity_distribution.png')

## 7. Figure 5: Category Distribution Heatmap

In [ ]:
def fig_category_heatmap(article_data, save_path=None):
    """Heatmap of concern categories across models."""
    model_cats = {}
    all_categories = set()
    
    for model_name, records in article_data.items():
        all_concerns = [c for r in records for c in r.get('structured_concerns', [])]
        if not all_concerns:
            continue
        counter = Counter(c.get('category', 'unknown') for c in all_concerns)
        total = sum(counter.values())
        model_cats[model_name] = {k: v / total for k, v in counter.items()}
        all_categories.update(counter.keys())
    
    if not model_cats:
        print('No category data.')
        return
    
    categories = sorted(all_categories)
    models = list(model_cats.keys())
    
    matrix = np.zeros((len(models), len(categories)))
    for i, m in enumerate(models):
        for j, c in enumerate(categories):
            matrix[i, j] = model_cats[m].get(c, 0)
    
    fig, ax = plt.subplots(figsize=(5, 2.5))
    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0)
    
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels([c.replace('_', ' ') for c in categories],
                       rotation=45, ha='right', fontsize=6)
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels(models)
    
    # Annotate cells
    for i in range(len(models)):
        for j in range(len(categories)):
            val = matrix[i, j]
            if val > 0.01:
                color = 'white' if val > 0.3 else 'black'
                ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                        fontsize=5, color=color)
    
    cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
    cbar.ax.set_ylabel('Proportion', fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path)
        fig.savefig(save_path.with_suffix('.pdf'))
    plt.show()

fig_category_heatmap(article_data, FIGURES_DIR / 'fig5_category_heatmap.png')

## 8. Figure 6: Generation Time Comparison

In [ ]:
def fig_generation_time(article_data, save_path=None):
    """Box plot of generation time per article."""
    plot_data = []
    model_names = []
    
    for model_name, records in article_data.items():
        times = [r.get('generation_time_s', 0) for r in records
                 if r.get('generation_time_s', 0) > 0]
        if not times:
            continue
        plot_data.append(times)
        model_names.append(model_name)
    
    if not plot_data:
        print('No timing data.')
        return
    
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    
    bp = ax.boxplot(plot_data, labels=model_names, patch_artist=True,
                    flierprops={'markersize': 2, 'alpha': 0.3},
                    medianprops={'color': '#E15759', 'linewidth': 1.0},
                    whiskerprops={'linewidth': 0.6},
                    capprops={'linewidth': 0.6},
                    boxprops={'linewidth': 0.6})
    
    for patch, name in zip(bp['boxes'], model_names):
        patch.set_facecolor(COLORS.get(name, '#4E79A7'))
        patch.set_alpha(0.6)
    
    ax.set_ylabel('Generation time (seconds)')
    ax.tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path)
        fig.savefig(save_path.with_suffix('.pdf'))
    plt.show()

fig_generation_time(article_data, FIGURES_DIR / 'fig6_generation_time.png')

## 9. Summary Table (LaTeX)

In [ ]:
def generate_latex_table(df):
    """Generate LaTeX table for publication."""
    print(r'\begin{table}[ht]')
    print(r'\centering')
    print(r'\caption{Performance comparison of fine-tuned models on the BioReview validation set (982 articles).}')
    print(r'\label{tab:model_comparison}')
    print(r'\begin{tabular}{lccccr}')
    print(r'\toprule')
    print(r'Model & Params & F1 & Precision & Recall & Concerns \\')
    print(r'\midrule')
    
    param_map = {
        'Qwen2.5-7B': '7B', 'Qwen3-8B': '8B', 'Qwen3.5-9B': '9B',
        'Qwen2.5-14B': '14B', 'DeepSeek-R1-14B': '14B',
        'GPT-4o-mini': '—', 'Gemini-2.5-Flash': '—',
    }
    
    sorted_df = df.sort_values('f1', ascending=False)
    best_f1 = sorted_df['f1'].max()
    
    for _, row in sorted_df.iterrows():
        model = row['model']
        params = param_map.get(model, '—')
        f1_str = f'\\textbf{{{row["f1"]:.4f}}}' if row['f1'] == best_f1 else f'{row["f1"]:.4f}'
        n_concerns = f'{int(row["n_tool_concerns"]):,}' if pd.notna(row.get('n_tool_concerns')) else '—'
        
        print(f'{model} & {params} & {f1_str} & '
              f'{row["precision"]:.4f} & {row["recall"]:.4f} & {n_concerns} \\\\')
    
    print(r'\bottomrule')
    print(r'\end{tabular}')
    print(r'\end{table}')

generate_latex_table(df)

## 10. Quick Statistics

In [ ]:
# ── Summary statistics ────────────────────────────────────────
print('='*60)
print('BioReview SFT Model Comparison Summary')
print('='*60)

for _, row in df.sort_values('f1', ascending=False).iterrows():
    print(f"\n{row['model']}:")
    print(f"  F1={row['f1']:.4f}  Prec={row['precision']:.4f}  Rec={row['recall']:.4f}")
    if pd.notna(row.get('n_tool_concerns')):
        avg_concerns = row['n_tool_concerns'] / row['processed'] if row['processed'] > 0 else 0
        print(f"  Concerns: {int(row['n_tool_concerns'])} total, {avg_concerns:.1f}/article")
        print(f"  Parse failures: {int(row.get('failed_parse', 0))}")
        if row.get('total_time'):
            print(f"  Inference time: {row['total_time']/60:.1f} min ({row['total_time']/row['processed']:.1f}s/article)")

print(f"\nBaseline: GPT-4o-mini F1=0.6962 (target to beat)")